# Evaluation Script: WebView Capabilities

This script produces results for Table 4.

### Connect to the MongoDB

In [ ]:
import tqdm
import pymongo

client = pymongo.MongoClient("mongodb://localhost:27017/")
db = client["webview"]
webviews_collection = db["webviews"]
dynamic_api_calls_collection = db["dynamic_api_calls"]
webview_breakdown_collection = db["webview_all_breakdown"] # this collection is created

def print_latex_macro(name: str, value: str):
    print(f"\\newcommand{{\\{name}}}{{{value}}}")

### Define functions for when a WebView offers a specific feature

In [3]:
def webview_allows_javascript(package_name: str, webview_id: int) -> bool:
    entries = dynamic_api_calls_collection.find({
        "source_package_name": package_name,
        "webview_id": webview_id,
        "api": "WEB_SETTINGS_SET_JAVA_SCRIPT_ENABLED"
    })
    
    sorted_entries = sorted(entries, key=lambda x: x.get("timestamp", 0))
    
    params = []
    for entry in sorted_entries:
        entry_param = entry.get("params", [])[0]
        params.append(entry_param)
    
    # check if all are True
    if len(params) == 0:
        return False
    
    if all(param == True for param in params):
        return True
    
    # if there are two values, we check if they happen within a short time frame (50ms)
    if len(params) == 2:
        time_diff = abs(sorted_entries[1].get("timestamp", 0) - sorted_entries[0].get("timestamp", 0))
        if time_diff < 0.05:
            # get the value of the last call
            last_value = params[-1]
            if last_value == True:
                return True
            else:
                return False
        
    
    if any(param == False for param in params) and any(param == True for param in params):
        return "CONFLICT_FALSE"
    else:
        return False
    
def webview_sets_javascript_bridge(package_name: str, webview_id: int) -> bool:
    count = dynamic_api_calls_collection.count_documents({
        "source_package_name": package_name,
        "webview_id": webview_id,
        "api": "ADD_JAVASCRIPT_INTERFACE"
    })
    sets_js_bridge = count > 0
    if sets_js_bridge:
        return True
    return False

def webview_sets_mc(package_name: str, webview_id: int) -> str:
    
    entries = dynamic_api_calls_collection.find({
        "source_package_name": package_name,
        "webview_id": webview_id,
        "api": "WEB_SETTINGS_SET_MIXED_CONTENT_MODE"
    })
    
    params = []
    for entry in entries:
        entry_param = entry.get("params", [])[0]
        params.append(entry_param)
        
    if len(params) == 0:
        return "NO_MC"
    
    if len(params) == 1:
        if params[0] == 0:
            return "ACTIVE_MC"
        elif params[0] == 2:
            return "PASSIVE_MC"
        elif params[0] == 1:
            return "NO_MC"
        else:
            raise ValueError(f"Unknown Mixed Content Mode value for webview {webview_id} in package {package_name}: {params[0]}")
        
    if not 1 in params:
        return "CONFLICT_PASSIVE"
    else:
        return "CONFLICT_NO_MC"
    
def webview_sets_allow_file_access_from_file(package_name: str, webview_id: int) -> bool:
    entries = dynamic_api_calls_collection.find({
        "source_package_name": package_name,
        "webview_id": webview_id,
        "api": "WEB_SETTINGS_SET_ALLOW_FILE_ACCESS_FROM_FILE_URLS"
    })
    params = []
    for entry in entries:
        entry_param = entry.get("params", [])[0]
        params.append(entry_param)
        
    if len(params) == 0:
        return False
    
    # check if all are True
    if all(param == True for param in params):
        return True
    # check if one is False, but one is True
    elif any(param == False for param in params) and any(param == True for param in params):
        return "CONFLICT_ALLOW_FILE_ACCESS"
    else:
        return False
    
def webview_sets_allow_universal_file_access(package_name: str, webview_id: int) -> bool:
    entries = dynamic_api_calls_collection.find({
        "source_package_name": package_name,
        "webview_id": webview_id,
        "api": "WEB_SETTINGS_SET_ALLOW_UNIVERSAL_ACCESS_FROM_FILE_URLS"
    })
    params = []
    for entry in entries:
        entry_param = entry.get("params", [])[0]
        params.append(entry_param)
        
    if len(params) == 0:
        return False
    
    # check if all are True
    if all(param == True for param in params):
        return True
    # check if one is False, but one is True
    elif any(param == False for param in params) and any(param == True for param in params):
        return "CONFLICT_ALLOW_UNIVERSAL_ACCESS"
    else:
        return False

### Define functions for when a WebView loads a specific resource

In [4]:
def webview_loads_subject_to_mixed_content(package_name: str, webview_id: int) -> bool:
    entries = dynamic_api_calls_collection.find({
        "source_package_name": package_name,
        "webview_id": webview_id,
        "api": "LOAD_URL",
        "params.0": {"$regex": "^https:"}
    })
    if len(list(entries)) > 0:
        return True
    
    results = dynamic_api_calls_collection.find({
        "source_package_name": package_name,
        "webview_id": webview_id,
        "api": "LOAD_DATA_WITH_BASE_URL",
        "params.0": {"$regex": "^https:"}
    })
    
    if len(list(results)) > 0:
        return True
    
    return False

def webview_loads_not_subject_to_mixed_content(package_name: str, webview_id: int) -> bool:
    entries = dynamic_api_calls_collection.find({
        "source_package_name": package_name,
        "webview_id": webview_id,
        "api": "LOAD_URL",
        "params.0": {"$not": {"$regex": "^https:"}}
    })
    
    if len(list(entries)) > 0:
        return True
    
    entries = dynamic_api_calls_collection.find({
        "source_package_name": package_name,
        "webview_id": webview_id,
        "api": "LOAD_DATA_WITH_BASE_URL",
        "params.0": {"$not": {"$regex": "^https:"}}
    })
    
    if len(list(entries)) > 0:
        return True

    entries = dynamic_api_calls_collection.find({
        "source_package_name": package_name,
        "webview_id": webview_id,
        "api": "LOAD_DATA",
    })
    
    if len(list(entries)) > 0:
        return True
    
    return False

def webview_sets_load_file(package_name: str, webview_id: int) -> bool:
    count = dynamic_api_calls_collection.count_documents({
        "source_package_name": package_name,
        "webview_id": webview_id,
        "api": {"$in": ["LOAD_URL"]},
        "params": {"$regex": "^file:"}
    })
    loads_file = count > 0
    if loads_file:
        return True
    
    
    entries = dynamic_api_calls_collection.find({
        "source_package_name": package_name,
        "webview_id": webview_id,
        "api": "LOAD_DATA_WITH_BASE_URL",
        "params.0": {"$regex": "^file:"}
    })
    if len(list(entries)) > 0:
        return True
    
    return False

In [6]:
instances = {}

for doc in tqdm.tqdm(webviews_collection.find(), total=webviews_collection.count_documents({})):
    package_name = doc["source_package_name"]
    webview_id = doc["webview_id"]
    id = doc["_id"]

    cached = webview_breakdown_collection.find_one({"_id": id})
    if cached is not None:
        instances[id] = {k: v for k, v in cached.items() if k != "_id"}
        continue

    allows_js = set()
    webview_allows_js = webview_allows_javascript(package_name, webview_id)
    allows_js.add(webview_allows_js)
            
    sets_js_bridge = set()
    webview_sets_js_bridge = webview_sets_javascript_bridge(package_name, webview_id)
    sets_js_bridge.add(webview_sets_js_bridge)
            
    allows_mc = set()
    webview_allows_mc = webview_sets_mc(package_name, webview_id)
    allows_mc.add(webview_allows_mc)
            
    allows_file_access_from_file = set()
    webview_allows_file_access_from_file = webview_sets_allow_file_access_from_file(package_name, webview_id)
    allows_file_access_from_file.add(webview_allows_file_access_from_file)
            
    allows_universal_file_access = set()
    webview_allows_universal_file_access = webview_sets_allow_universal_file_access(package_name, webview_id)
    allows_universal_file_access.add(webview_allows_universal_file_access)
            
    loads_file = None
    webview_loads_file = webview_sets_load_file(package_name, webview_id)
    loads_file = webview_loads_file
            
    
    url_subject_to_mixed_content_loaded = None
    webview_url_subject_to_mixed_content_loaded = webview_loads_subject_to_mixed_content(package_name, webview_id)
    url_subject_to_mixed_content_loaded = webview_url_subject_to_mixed_content_loaded
            
    url_not_subject_to_mixed_content_loaded = None
    webview_url_not_subject_to_mixed_content_loaded = webview_loads_not_subject_to_mixed_content(package_name, webview_id)
    url_not_subject_to_mixed_content_loaded = webview_url_not_subject_to_mixed_content_loaded
    
            
    record = {
        "webview_id": webview_id, 
        "package_name": package_name,
        "allows_javascript": list(allows_js),
        "sets_javascript_bridge": list(sets_js_bridge),
        "allows_mc": list(allows_mc),
        "allows_file_access_from_file": list(allows_file_access_from_file),
        "allows_universal_file_access": list(allows_universal_file_access),
        "loads_file": loads_file,
        "url_subject_to_mixed_content_loaded": url_subject_to_mixed_content_loaded,
        "url_not_subject_to_mixed_content_loaded": url_not_subject_to_mixed_content_loaded,
    }
    webview_breakdown_collection.insert_one({"_id": id, **record})
    instances[id] = record

100%|██████████| 658894/658894 [4:49:58<00:00, 37.87it/s]     


### Generate the WebView breakdown table

In [ ]:
CONFLICT_TIERS = [
    (0.005,  r"\dags{*}"),
    (0.025, r"\dags{\dag}"),
    (0.99,  r"\dags{\ddag}"),
]
CONSERVATIVE_FIELDS = {
    "allows_javascript",
    "allows_file_access_from_file",
    "allows_universal_file_access",
}

# MC conflicts (CONFLICT_PASSIVE, CONFLICT_NO_MC) are folded into "Blocks MC"
# (conservative: if we cannot tell, do not count the WebView as allowing MC).
MC_CONFLICT_VALUES = ["CONFLICT_PASSIVE", "CONFLICT_NO_MC"]

MC_MODES = [
    ("Allows active MC",  [["ACTIVE_MC"]]),
    ("Allows passive MC", [["PASSIVE_MC"]]),
    ("Blocks MC",         [["NO_MC"], ["CONFLICT_PASSIVE"], ["CONFLICT_NO_MC"]]),
]

CAPABILITIES = [
    (r"\jsEnabled",           "allows_javascript"),
    (r"\jsBridge",            "sets_javascript_bridge"),
    (r"\fileAccess",          "allows_file_access_from_file"),
    (r"\universalFileAccess", "allows_universal_file_access"),
]

CONFLICT_MARKERS = [
    "CONFLICT_FALSE",
    "CONFLICT_ALLOW_FILE_ACCESS",
    "CONFLICT_ALLOW_UNIVERSAL_ACCESS",
    "CONFLICT_PASSIVE",
    "CONFLICT_NO_MC",
]

SUBJECT     = {"url_subject_to_mixed_content_loaded": True}
NOT_SUBJECT = {"url_not_subject_to_mixed_content_loaded": True}
LOADS_FILE  = {"loads_file": True}


def combine(*queries):
    merged = {}
    for q in queries:
        merged.update(q)
    return merged


def count_groups(query):
    return webview_breakdown_collection.count_documents(query)


def mc_query(values):
    # values is a list of allows_mc array shapes to accept (e.g. [["NO_MC"], ["CONFLICT_PASSIVE"]]).
    return {"allows_mc": {"$in": values}}


def cap_query(field, value=True):
    # Strict: the per-WebView value list must be exactly [value].
    return {field: [value]}


def conflict_query(field):
    # WebView's capability list contains a CONFLICT_* marker.
    return {field: {"$in": CONFLICT_MARKERS}}


TOTAL = count_groups({})
N_MC_CONFLICTS = count_groups({"allows_mc": {"$in": [[v] for v in MC_CONFLICT_VALUES]}})
PCT_MC_CONFLICTS = 0.0 if not TOTAL else 100.0 * N_MC_CONFLICTS / TOTAL


def fmt_n(n):
    return f"{n:,}"


def fmt_bar(n, denom):
    pct = 0.0 if not denom else 100.0 * n / denom
    return rf"\databar{{{pct:.1f}}}{{{pct:.1f}\%}}"


def conflict_dagger(field, n_conflict, n_row):
    if field not in CONSERVATIVE_FIELDS or n_conflict <= 0 or n_row <= 0:
        return ""
    ratio = n_conflict / n_row
    for threshold, marker in CONFLICT_TIERS:
        if ratio < threshold:
            return marker
    return ""


def row_wv(label, query):
    n_row = count_groups(query)
    cells = []
    for _, field in CAPABILITIES:
        n = count_groups(combine(query, cap_query(field, True)))
        n_conflict = count_groups(combine(query, conflict_query(field)))
        dagger = conflict_dagger(field, n_conflict, n_row)
        cells.append(f"{fmt_n(n)}{dagger}")
        cells.append(fmt_bar(n, n_row))
    cells.append(fmt_n(n_row))
    cells.append(fmt_bar(n_row, TOTAL))
    return f"    {label} & {' & '.join(cells)} \\\\"


def build_table(row_func, super_header, caption, label):
    n_cols = len(CAPABILITIES) + 1  # capabilities + Total
    n_data = 2 * n_cols
    # Each data column is a (right-aligned number, centered bar) pair.
    col_spec = "p{4.2cm} " + " ".join(["r", "c"] * n_cols)

    header_groups = [rf"\multicolumn{{2}}{{c}}{{{h}}}" for h, _ in CAPABILITIES]
    header_groups.append(r"\multicolumn{2}{c}{\textbf{Total}}")
    cmidrules = " ".join(
        rf"\cmidrule(lr){{{2 + 2 * i}-{3 + 2 * i}}}" for i in range(n_cols)
    )

    lines = [
        r"\begin{table*}[ht]",
        r"  \centering",
        r"  \footnotesize",
        r"  \setlength{\tabcolsep}{4pt}",
        r"  \renewcommand{\arraystretch}{1.2}",
        rf"  \begin{{tabular}}{{{col_spec}}}",
        r"    \toprule",
        r"    \multirow{2}{4.2cm}{\textbf{MC Configuration / \\Loaded Content}} & "
        rf"\multicolumn{{{n_data}}}{{c}}{{\textbf{{{super_header}}}}} \\",
        rf"    \cmidrule(l){{2-{1 + n_data}}}",
        rf"    & {' & '.join(header_groups)} \\",
        rf"    {cmidrules}",
    ]

    for i, (mode_label, mc_values) in enumerate(MC_MODES):
        if i:
            lines.append(r"    \addlinespace")
        q = mc_query(mc_values)
        lines.append(row_func(r"\textbf{\emph{" + mode_label + r"}}", q))
        lines.append(row_func(r"\enskip Loads content subject to MCP",     combine(q, SUBJECT)))
        lines.append(row_func(r"\enskip Loads content not subject to MCP", combine(q, NOT_SUBJECT)))

    lines += [
        r"    \midrule",
        row_func(r"\textbf{Loads \code{file://}}", LOADS_FILE),
        r"    \midrule",
        row_func(r"\textbf{Total}", {}),
        r"    \bottomrule",
        r"  \end{tabular}",
        rf"  \caption{{{caption}}}",
        rf"  \label{{{label}}}",
        r"\end{table*}",
    ]
    return "\n".join(lines)


WV_CAPTION = (
    r"WebView instances (one per observed WebView) by Mixed Content (MC) "
    r"configuration and loading behaviour. Percentages (\%) indicate row "
    r"shares, except in the `Total' column which shows overall share. "
    r"Capabilities require consistent enablement across all observed API "
    r"calls for that WebView; WebViews with conflicting calls default to "
    r"disabled. Daggers denote the row share excluded by this conservative "
    r"choice (\textsuperscript{*}\,$< 0.5$\%, \textsuperscript{\dag}\,$< 2.5$\%). "
    rf"WebViews with conflicting MC settings ({fmt_n(N_MC_CONFLICTS)}, "
    rf"{PCT_MC_CONFLICTS:.2f}\% of all WebViews) are counted in `Blocks MC' "
    r"(conservative: not counted as allowing MC). "
    r"Legend: \jsEnabled~JS enabled; \jsBridge~JS Bridge exposed; "
    r"\fileAccess~file access from file URLs; \universalFileAccess~universal "
    r"access from file URLs. "
    r"\textit{Note: Loading sub-categories are not mutually exclusive.}"
)

print(build_table(row_wv,  "WebView Instances", WV_CAPTION,  "tab:webview-mc-wv"))

print_latex_macro("webViewsAmbiguityPercentage", f"{PCT_MC_CONFLICTS:.2f}")


\begin{table*}[ht]
  \centering
  \footnotesize
  \setlength{\tabcolsep}{4pt}
  \renewcommand{\arraystretch}{1.2}
  \begin{tabular}{p{4.2cm} r c r c r c r c r c}
    \toprule
    \multirow{2}{4.2cm}{\textbf{MC Configuration / \\Loaded Content}} & \multicolumn{10}{c}{\textbf{WebView Instances}} \\
    \cmidrule(l){2-11}
    & \multicolumn{2}{c}{\jsEnabled} & \multicolumn{2}{c}{\jsBridge} & \multicolumn{2}{c}{\fileAccess} & \multicolumn{2}{c}{\universalFileAccess} & \multicolumn{2}{c}{\textbf{Total}} \\
    \cmidrule(lr){2-3} \cmidrule(lr){4-5} \cmidrule(lr){6-7} \cmidrule(lr){8-9} \cmidrule(lr){10-11}
    \textbf{\emph{Allows active MC}} & 24,094\dags{\dag} & \databar{99.1}{99.1\%} & 16,327 & \databar{67.1}{67.1\%} & 3,800\dags{*} & \databar{15.6}{15.6\%} & 5,859\dags{*} & \databar{24.1}{24.1\%} & 24,323 & \databar{3.7}{3.7\%} \\
    \enskip Loads content subject to MCP & 10,548\dags{\dag} & \databar{98.8}{98.8\%} & 7,759 & \databar{72.7}{72.7\%} & 874\dags{*} & \databar{8.2}{8.2\%} & 1